# Bitcoin price prediction with stable evolutionary training
This notebook calls the same tested modules as the CLI. Run top-to-bottom. Defaults below are short for interactive verification; set `hours=5` for the full run. Not financial advice.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.hardware import detect
print(json.dumps(detect('data'), indent=2))

## Download and cache one year of 1-minute OHLCV

In [ ]:
from src.data import download_year, prepare
raw = download_year('data/btcusdt_1m.parquet', days=365)
features = prepare(raw, 'data/features.npz', horizons=(1,5,15,60,240,720))
raw, features

## Inspect prepared arrays

In [ ]:
import numpy as np
z=np.load(features, mmap_mode='r')
print('X', z['X'].shape, 'y', z['y'].shape)
print('features:', z['features'].tolist())
print('horizons:', z['horizons'].tolist())

## Train, evaluate, checkpoint, and save
Change `hours` to `5.0` for the requested production budget. For maximum multi-GPU throughput, use `./run_multigpu.sh` in a terminal; notebook kernels should not spawn `torchrun`.

In [ ]:
from argparse import Namespace
from src.train import main
cfg = Namespace(data='data/features.npz', raw='data/btcusdt_1m.parquet', horizons='1,5,15,60,240,720', hours=0.05, population=4, mutation_rate=.03, crossover_rate=.08, elite_fraction=.5, epochs_per_generation=1, steps_per_candidate=100, batch_size=128, accumulate=1, sequence=240, lr=3e-4, checkpoint_dir='checkpoints', model_dir='models', checkpoint_minutes=10, seed=42, resume=True, download=False, island_id=0)
main(cfg)

## Load the saved result

In [ ]:
import torch
result=torch.load('models/best_island0.pt', map_location='cpu', weights_only=False)
result['metrics']